# 02 — Preprocessing

Load raw JSON, extract snapshot date from filename, cast date columns, normalise plural/singular schema splits, write clean Parquet.

## 1. Start Spark session

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('FB_API') \
    .getOrCreate()

print('Master:', spark.sparkContext.master)
print('Spark version:', spark.version)

## 2. Load JSON and extract snapshot date from filename

In [ ]:
from pyspark.sql.functions import input_file_name, regexp_extract, to_date, substring

DATA_PATH = '/data/ProjectDatasetFacebookAU/'

df = spark.read.json(DATA_PATH)
df = df.withColumn('filename', input_file_name())
df = df.withColumn(
    'snapshot_date',
    to_date(regexp_extract('filename', r'(\d{8})', 1), 'yyyyMMdd')
)

## 3. Cast date columns

`ad_creation_time` has two formats across the dataset: `yyyy-MM-dd` (most records) and `yyyy-MM-ddTHH:mm:ss+0000` (early records). Slicing the first 10 characters handles both safely.

Same treatment applied to `ad_delivery_start_time` and `ad_delivery_stop_time`.

In [ ]:
date_cols = ['ad_creation_time', 'ad_delivery_start_time', 'ad_delivery_stop_time']

for c in date_cols:
    df = df.withColumn(
        c.replace('_time', '_date'),
        to_date(substring(c, 1, 10), 'yyyy-MM-dd')
    )

## 4. Normalise schema

The API renamed several fields from singular strings to arrays in mid-2022 (Graph API v13.0). Use `coalesce` to merge each pair into a single array column. Also drop `filename` and the original split columns — the SQL SELECT defines the final schema explicitly.

In [ ]:
df.createOrReplaceTempView('ads')

df = spark.sql("""
    SELECT
        id,
        page_id,
        page_name,
        funding_entity,
        bylines,
        snapshot_date,
        ad_creation_date,
        ad_delivery_start_date,
        ad_delivery_stop_date,
        coalesce(ad_creative_bodies,            array(ad_creative_body))            AS creative_bodies,
        coalesce(ad_creative_link_captions,     array(ad_creative_link_caption))    AS creative_link_captions,
        coalesce(ad_creative_link_descriptions, array(ad_creative_link_description)) AS creative_link_descs,
        coalesce(ad_creative_link_titles,       array(ad_creative_link_title))      AS creative_link_titles,
        impressions,
        spend,
        currency,
        languages,
        publisher_platforms,
        demographic_distribution,
        coalesce(delivery_by_region, region_distribution) AS delivery_by_region,
        estimated_audience_size,
        ad_snapshot_url
    FROM ads
""")

df.printSchema()